In [ ]:
import json
import os
import numpy as np
from scipy.stats import wilcoxon, shapiro, ttest_rel
from collections import defaultdict
from hcr import HTRInference
import torch
from test_trocr_standalone import load_model, predict_one
from original_trocr import load, predict

CHARSET = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
               "0123456789 $#&%*.,;:¡!¿?«»-_'\"()[]áéíóúüñÁÉÍÓÚÜÑ")
IMG_HEIGHT = 64
IMG_WIDTH  = 256
BATCH_SIZE = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def edit_distance(s1, s2):
    """Distancia de Levenshtein entre dos secuencias."""
    m, n = len(s1), len(s2)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]

IMAGE_FOLDER = "final"
JSON_PATH    = "test.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    el_json = json.load(f)

processor1, model1 = load()
processor, model = load_model('trocr_es_finetuned', 4, 128)

image_entries = []
for entry in sorted(os.scandir(IMAGE_FOLDER), key=lambda e: e.name):
    if entry.is_file():
        image_entries.append((entry.name, entry.path))
    elif entry.is_dir():
        for sub in sorted(os.scandir(entry.path), key=lambda e: e.name):
            if sub.is_file():
                image_entries.append((f"{entry.name}/{sub.name}", sub.path))

refs       = []
hyps_best  = []
hyps_early = []

print("Evaluando ambos modelos...\n")

for key_ext, image_path in image_entries:
    key_no_ext = os.path.splitext(key_ext)[0]
    clave = key_ext if key_ext in el_json else (key_no_ext if key_no_ext in el_json else None)
    if clave is None:
        continue

    reference = el_json[clave]
    pred_best = predict(image_path, model1, processor1)
    print(pred_best)
    text, dt = predict_one(processor, model, image_path)
    print(text)

    refs.append(reference)
    hyps_best.append(pred_best)
    hyps_early.append(text)

def cer_global(preds, targets):
    total_edit = sum(edit_distance(p, t) for p, t in zip(preds, targets))
    total_chars = sum(len(t) for t in targets)
    return total_edit / max(total_chars, 1)

def wer_global(preds, targets):
    total_edit = sum(edit_distance(p.split(), t.split()) for p, t in zip(preds, targets))
    total_words = sum(len(t.split()) for t in targets)
    return total_edit / max(total_words, 1)

def cer_por_muestra(preds, targets):
    return np.array([
        edit_distance(p, t) / max(len(t), 1)
        for p, t in zip(preds, targets)
    ])

cer_arr_best  = cer_por_muestra(hyps_best,  refs)
cer_arr_early = cer_por_muestra(hyps_early, refs)

ALPHA = 0.05

diffs = cer_arr_best - cer_arr_early

# Shapiro-Wilk requiere al menos 3 muestras y falla si todas las diferencias
# son idénticas (varianza cero), por lo que añadimos un guard.
if len(diffs) < 3 or np.all(diffs == 0):
    shapiro_stat, shapiro_p = None, None
    es_normal = False          # sin datos suficientes → no asumimos normalidad
else:
    shapiro_stat, shapiro_p = shapiro(diffs)
    es_normal = shapiro_p >= ALPHA   # H0: la distribución ES normal

if es_normal:
    test_nombre = "t de Student pareada"
    # ttest_rel compara directamente las dos muestras pareadas
    stat, p_value = ttest_rel(cer_arr_best, cer_arr_early)
else:
    test_nombre = "Wilcoxon signed-rank"
    if np.all(diffs == 0):
        stat, p_value = None, 1.0
    else:
        stat, p_value = wilcoxon(cer_arr_best, cer_arr_early)

n = len(refs)
print("═" * 60)
print("  COMPARACIÓN DE MODELOS — TEST SET REAL")
print("═" * 60)
print(f"  Muestras evaluadas  : {n}")
print()
print(f"  {'Métrica':<22} {'trocr_original':>14} {'trocr_finetuned':>14}")
print(f"  {'─'*50}")
print(f"  {'CER global':<22} {cer_global(hyps_best, refs):>13.4f}  "
      f"{cer_global(hyps_early, refs):>13.4f}")
print(f"  {'WER global':<22} {wer_global(hyps_best, refs):>13.4f}  "
      f"{wer_global(hyps_early, refs):>13.4f}")
print(f"  {'CER medio ± std':<22} "
      f"{cer_arr_best.mean():>7.4f}±{cer_arr_best.std():.4f}  "
      f"{cer_arr_early.mean():>7.4f}±{cer_arr_early.std():.4f}")
print(f"  {'CER mediana':<22} {np.median(cer_arr_best):>13.4f}  "
      f"{np.median(cer_arr_early):>13.4f}")
print()

# — Normalidad —
print("  ── Test de normalidad (Shapiro-Wilk sobre diferencias pareadas) ──")
if shapiro_stat is None:
    print("  Shapiro-Wilk: no aplicable (n < 3 o diferencias constantes)")
    print(f"  → Distribución asumida: NO NORMAL")
else:
    print(f"  Shapiro-Wilk  W={shapiro_stat:.4f}  p={shapiro_p:.4f}")
    print(f"  → Distribución de diferencias: "
          f"{'NORMAL' if es_normal else 'NO NORMAL'} (α={ALPHA})")
print()

# — Test de hipótesis —
print(f"  ── Test seleccionado: {test_nombre} ──")
print(f"  stat={stat}  p={p_value:.4f}")
print()

if p_value < ALPHA:
    ganador = ("trocr_original"
               if cer_arr_best.mean() < cer_arr_early.mean()
               else "trocr_finetuned")
    print(f"  → Diferencia SIGNIFICATIVA (p < {ALPHA})")
    print(f"  → Mejor modelo: {ganador}")
else:
    ganador = ("trocr_original"
               if cer_arr_best.mean() <= cer_arr_early.mean()
               else "trocr_finetuned")
    print(f"  → Sin diferencia significativa (p ≥ {ALPHA})")
    print(f"  → Desempate por CER medio: {ganador}")

print("═" * 60)

In [ ]:
import json
import os
import numpy as np
from scipy.stats import wilcoxon
from collections import defaultdict
from hcr import HTRInference
import torch
from test_trocr_standalone import load_model, predict_one
from original_trocr import load, predict

CHARSET = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
               "0123456789 $#&%*.,;:¡!¿?«»-_'\"()[]áéíóúüñÁÉÍÓÚÜÑ")
IMG_HEIGHT = 64          # altura de precomputación y entrada al modelo
IMG_WIDTH  = 256         # FIX 3: era 320 → 256  (suficiente para palabras)
BATCH_SIZE = 128         # FIX 5: era 96 → 128   (mejor saturación GPU con AMP)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def edit_distance(s1, s2):
    """Distancia de Levenshtein entre dos secuencias."""
    m, n = len(s1), len(s2)
    dp = np.zeros((m + 1, n + 1), dtype=int)

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],    # borrar
                                       dp[i][j-1],    # insertar
                                       dp[i-1][j-1])  # sustituir
    return dp[m][n]

IMAGE_FOLDER = "final"
JSON_PATH    = "test.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    el_json = json.load(f)

processor1, model1 = load()
processor, model = load_model('trocr_es_finetuned', 4, 128)

image_entries = []
for entry in sorted(os.scandir(IMAGE_FOLDER), key=lambda e: e.name):
    if entry.is_file():
        image_entries.append((entry.name, entry.path))
    elif entry.is_dir():
        for sub in sorted(os.scandir(entry.path), key=lambda e: e.name):
            if sub.is_file():
                image_entries.append((f"{entry.name}/{sub.name}", sub.path))

refs        = []
hyps_best   = []
hyps_early  = []

print("Evaluando ambos modelos...\n")

for key_ext, image_path in image_entries:
    key_no_ext = os.path.splitext(key_ext)[0]
    clave = key_ext if key_ext in el_json else (key_no_ext if key_no_ext in el_json else None)
    if clave is None:
        continue

    reference = el_json[clave]
    # pred_best  = inferencer_best.predict(image_path)
    pred_best = predict(image_path, model1, processor1)
    print(pred_best)
    text, dt = predict_one(processor, model, image_path)
    print(text)

    refs.append(reference)
    hyps_best.append(pred_best)
    hyps_early.append(text)

def cer_global(preds, targets):
    total_edit = sum(edit_distance(p, t) for p, t in zip(preds, targets))
    total_chars = sum(len(t) for t in targets)
    return total_edit / max(total_chars, 1)

def wer_global(preds, targets):
    total_edit = sum(edit_distance(p.split(), t.split()) for p, t in zip(preds, targets))
    total_words = sum(len(t.split()) for t in targets)
    return total_edit / max(total_words, 1)

def cer_por_muestra(preds, targets):
    return np.array([
        edit_distance(p, t) / max(len(t), 1)
        for p, t in zip(preds, targets)
    ])

cer_arr_best  = cer_por_muestra(hyps_best,  refs)
cer_arr_early = cer_por_muestra(hyps_early, refs)

diffs = cer_arr_best - cer_arr_early
if np.all(diffs == 0):
    stat, p_value = None, 1.0
else:
    stat, p_value = wilcoxon(cer_arr_best, cer_arr_early)

n = len(refs)
print("═" * 55)
print("  COMPARACIÓN DE MODELOS — TEST SET REAL")
print("═" * 55)
print(f"  Muestras evaluadas  : {n}")
print()
print(f"  {'Métrica':<22} {'trocr_original':>12} {'trocr_finetuned':>12}")
print(f"  {'─'*46}")
print(f"  {'CER global':<22} {cer_global(hyps_best, refs):>11.4f}  "
      f"{cer_global(hyps_early, refs):>11.4f}")
print(f"  {'WER global':<22} {wer_global(hyps_best, refs):>11.4f}  "
      f"{wer_global(hyps_early, refs):>11.4f}")
print(f"  {'CER medio ± std':<22} "
      f"{cer_arr_best.mean():>6.4f}±{cer_arr_best.std():.4f}  "
      f"{cer_arr_early.mean():>6.4f}±{cer_arr_early.std():.4f}")
print(f"  {'CER mediana':<22} {np.median(cer_arr_best):>11.4f}  "
      f"{np.median(cer_arr_early):>11.4f}")
print()
print(f"  Wilcoxon  stat={stat}  p={p_value:.4f}")
print()

ALPHA = 0.05
if p_value < ALPHA:
    ganador = "trocr_original" if cer_arr_best.mean() < cer_arr_early.mean() else "trocr_finetuned"
    print(f"  → Diferencia SIGNIFICATIVA (p<{ALPHA})")
    print(f"  → Usar: {ganador}")
else:
    # Sin significancia estadística: elegir el de menor CER medio como desempate
    ganador = "trocr_original" if cer_arr_best.mean() <= cer_arr_early.mean() else "trocr_finetuned"
    print(f"  → Sin diferencia significativa (p≥{ALPHA})")
    print(f"  → Desempate por CER medio: usar {ganador}")

print("═" * 55)